In [1]:
!pip install wordcloud

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
%matplotlib inline 
import string
from wordcloud import WordCloud

import nltk 
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

nltk.download("stopwords")
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\lokes\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\lokes\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [3]:
df = pd.read_csv("./spam.csv")

df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [4]:
df.drop(columns=['Unnamed: 2','Unnamed: 3','Unnamed: 4'], inplace=True)

In [5]:
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
df.rename(columns={'v1':'target','v2':'text'},inplace = True)

In [7]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df['target'] = encoder.fit_transform(df['target'])

In [8]:
df.head()

,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [9]:
df.duplicated().sum()

np.int64(403)

In [10]:
df.drop_duplicates(keep='first',inplace = True)

In [11]:
def transform_text(text):

    text = text.lower()

    text = nltk.word_tokenize(text)

    y = []
    for i in text:
        if i.isalnum():
            y.append(i)

    text = y[:]
    y.clear()

    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i)

    text = y[:]
    y.clear()

    ps = PorterStemmer()
    for i in text:
        y.append(ps.stem(i))

    return " ".join(y)


In [12]:
transform_text("Go Untill")

'go until'

In [13]:
df['transformed_text'] = df['text'].apply(transform_text)

In [14]:
df.head()

,target,text,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,0,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


In [15]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

tfid = TfidfVectorizer(max_features = 500)

In [16]:
x = tfid.fit_transform(df['transformed_text']).toarray()
y = df['target'].values

In [17]:
from sklearn.model_selection import train_test_split

x_train,x_test, y_train, y_test = train_test_split(x,y,test_size= 0.2, random_state = 2)

In [20]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, BaggingClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

In [21]:
svc = SVC(kernel='sigmoid',gamma=1.0)
knn = KNeighborsClassifier()
mnb = MultinomialNB()
dtc = DecisionTreeClassifier(max_depth=5)
lrc = LogisticRegression(solver='liblinear',penalty='l1')
rfc = RandomForestClassifier(n_estimators=50, random_state=2)
adc = AdaBoostClassifier(n_estimators=50, random_state=2)
bc = BaggingClassifier(n_estimators=50, random_state=2)
etc = ExtraTreesClassifier(n_estimators=50, random_state=2)
gbdt = GradientBoostingClassifier(n_estimators=50, random_state=2)
xgb = XGBClassifier(n_estimators = 50, random_state = 2)

In [46]:
clfs = {"svc": svc ,
"knn": knn ,
"mnb": mnb ,
"dtc": dtc ,
"lrc": lrc ,
"rfc": rfc ,
"adc": adc ,
"bc":   bc ,
"etc": etc ,
"gbdt": gbdt,
"xgb": xgb 
}

clfs.items()

dict_items([('svc', SVC(gamma=1.0, kernel='sigmoid')), ('knn', KNeighborsClassifier()), ('mnb', MultinomialNB()), ('dtc', DecisionTreeClassifier(max_depth=5)), ('lrc', LogisticRegression(penalty='l1', solver='liblinear')), ('rfc', RandomForestClassifier(n_estimators=50, random_state=2)), ('adc', AdaBoostClassifier(random_state=2)), ('bc', BaggingClassifier(n_estimators=50, random_state=2)), ('etc', ExtraTreesClassifier(n_estimators=50, random_state=2)), ('gbdt', GradientBoostingClassifier(n_estimators=50, random_state=2)), ('xgb', XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_c

In [41]:
from sklearn.metrics import accuracy_score, precision_score

def train_classifier(clfs, x_train, y_train, x_test, y_test):
    clfs.fit(x_train, y_train)
    y_pred = clfs.predict(x_test)
    accuracy = accuracy_score(y_test,y_pred)
    precision = precision_score(y_test, y_pred)
    return accuracy, precision


In [48]:
accuracy = []
precision = []

for name, clf in clfs.items():

    current_accuracy,current_precision = train_classifier(clf, x_train,y_train,x_test,y_test)
    print()
    print("For: ",name)
    print("Accuracy: ",current_accuracy)
    print("precision: ", current_precision)

    accuracy.append(current_accuracy)
    precision.append(current_precision)


For:  svc
Accuracy:  0.9671179883945842
precision:  0.9333333333333333

For:  knn
Accuracy:  0.9274661508704062
precision:  1.0

For:  mnb
Accuracy:  0.9709864603481625
precision:  0.9655172413793104

For:  dtc
Accuracy:  0.937137330754352
precision:  0.9010989010989011

For:  lrc
Accuracy:  0.9622823984526112
precision:  0.9541284403669725


c:\Users\lokes\miniconda3\envs\mlops\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\lokes\miniconda3\envs\mlops\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(



For:  rfc
Accuracy:  0.9700193423597679
precision:  0.9421487603305785

For:  adc
Accuracy:  0.9235976789168279
precision:  0.8734177215189873

For:  bc
Accuracy:  0.9622823984526112
precision:  0.9024390243902439

For:  etc
Accuracy:  0.9709864603481625
precision:  0.921875

For:  gbdt
Accuracy:  0.9497098646034816
precision:  0.93

For:  xgb
Accuracy:  0.9690522243713733
precision:  0.9568965517241379


In [4]:
def transform_text(text):
    """Transform the input text by converting it to lowercase, tokenize, removing stopwords and punctuations, and stemming"""

    ps = PorterStemmer()

    text = text.lower()

    text = nltk.word_tokenize(text)

    text = [word for word in text if word.isalnum()]

    text = [word for word in text if word not in stopwords.words('english') and word not in string.punctuation]

    text = [ps.stem(word) for word in text]

    return " ".join(text)

In [5]:
transform_text("fsjdgfads dfasdfewpo")

'fsjdgfad dfasdfewpo'